Loading The Dataset From Kaggle

In [41]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/raw/train.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (45593, 20)

Columns: ['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']

First 5 rows:


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


Columns Inside Dataset And No. Of Null values present in each column along with datatype

In [42]:
# Cleaning column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

['id', 'delivery_person_id', 'delivery_person_age', 'delivery_person_ratings', 'restaurant_latitude', 'restaurant_longitude', 'delivery_location_latitude', 'delivery_location_longitude', 'order_date', 'time_orderd', 'time_order_picked', 'weatherconditions', 'road_traffic_density', 'vehicle_condition', 'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival', 'city', 'time_taken(min)']

Data types:
id                              object
delivery_person_id              object
delivery_person_age             object
delivery_person_ratings         object
restaurant_latitude            float64
restaurant_longitude           float64
delivery_location_latitude     float64
delivery_location_longitude    float64
order_date                      object
time_orderd                     object
time_order_picked               object
weatherconditions               object
road_traffic_density            object
vehicle_condition                int64
type_of_order                   object
t

Synthetically add rest of the columns required for the model

Extracting hour column

In [43]:
print(df['time_orderd'].head(10))

0    11:30:00
1    19:45:00
2    08:30:00
3    18:00:00
4    13:30:00
5    21:20:00
6    19:15:00
7    17:25:00
8    20:55:00
9    21:55:00
Name: time_orderd, dtype: object


In [44]:
# Hour extract from time_orderd 
df['hour'] = pd.to_datetime(
    df['time_orderd'],
    format='%H:%M:%S',
    errors='coerce'
).dt.hour

print("Hour extraction done:")
print(df['hour'].value_counts().sort_index())

Hour extraction done:
hour
0.0      430
8.0     1818
9.0     1947
10.0    1991
11.0    1962
12.0     892
13.0     784
14.0     791
15.0     873
16.0     709
17.0    4278
18.0    4480
19.0    4595
20.0    4539
21.0    4686
22.0    4576
23.0    4511
Name: count, dtype: int64


Hours 8-11  → Morning orders       
Hours 17-23 → Evening/Night peak   
Hour 0      → 430 rows — Midnight  
Hours 1-7   → Completely missing   
India mein 1 AM - 7 AM delivery almost zero hoti hai — toh missing hona normal hai.

Filling NaN values in hour column with mode

In [45]:
print("NaN in hour:", df['hour'].isnull().sum())
df['hour'] = df['hour'].fillna(df['hour'].mode()[0]).astype(int)
print("\nHour range:", df['hour'].min(), "to", df['hour'].max())
print("Total rows:", len(df))

NaN in hour: 1731

Hour range: 0 to 23
Total rows: 45593


Column type          Best fill method

Continuous number  → Median
  (salary, price)    (outliers se safe)

Pattern based      → Mode
  (hour, day)        (frequency match karta hai)

Categorical        → Mode
  (weather, zone)    (most common category)

Extracting day_of_week,is_weekend,is_lunch_time,is_dinner_time columns

In [46]:
print(df['order_date'].head(10))
print("\nData type:", df['order_date'].dtype)

0    19-03-2022
1    25-03-2022
2    19-03-2022
3    05-04-2022
4    26-03-2022
5    11-03-2022
6    04-03-2022
7    14-03-2022
8    20-03-2022
9    12-02-2022
Name: order_date, dtype: object

Data type: object


In [47]:
# converting order_date to proper date format  
df['order_date'] = pd.to_datetime(
    df['order_date'],
    format='%d-%m-%Y',
    errors='coerce'
)

df['day_of_week'] = df['order_date'].dt.dayofweek
# 0=Monday, 1=Tuesday ... 6=Sunday

# Weekend flag
df['is_weekend'] = df['day_of_week'].apply(
    lambda x: 1 if x >= 5 else 0
)

# Extracting Lunch and dinner flag from hour
df['is_lunch_time'] = df['hour'].apply(
    lambda x: 1 if 12 <= x <= 14 else 0
)
df['is_dinner_time'] = df['hour'].apply(
    lambda x: 1 if 19 <= x <= 22 else 0
)

print("Day of week distribution:")
print(df['day_of_week'].value_counts().sort_index())
print("\nWeekend distribution:")
print(df['is_weekend'].value_counts())
print("\nSample:")
print(df[['order_date', 'day_of_week',
          'is_weekend', 'is_lunch_time',
          'is_dinner_time']].head(10))

Day of week distribution:
day_of_week
0    6207
1    6375
2    7093
3    6348
4    7031
5    6290
6    6249
Name: count, dtype: int64

Weekend distribution:
is_weekend
0    33054
1    12539
Name: count, dtype: int64

Sample:
  order_date  day_of_week  is_weekend  is_lunch_time  is_dinner_time
0 2022-03-19            5           1              0               0
1 2022-03-25            4           0              0               1
2 2022-03-19            5           1              0               0
3 2022-04-05            1           0              0               0
4 2022-03-26            5           1              1               0
5 2022-03-11            4           0              0               1
6 2022-03-04            4           0              0               1
7 2022-03-14            0           0              0               0
8 2022-03-20            6           1              0               1
9 2022-02-12            5           1              0               1


Extracting weatherconditions column

In [48]:

print(df['weatherconditions'].value_counts())

# Step 1 — remove "conditions " from weatherconditions
df['weather'] = df['weatherconditions'].str.replace(
    'conditions ', '', regex=False
).str.strip()

print("After cleaning:")
print(df['weather'].value_counts(dropna=False))

weatherconditions
conditions Fog           7654
conditions Stormy        7586
conditions Cloudy        7536
conditions Sandstorms    7495
conditions Windy         7422
conditions Sunny         7284
conditions NaN            616
Name: count, dtype: int64
After cleaning:
weather
Fog           7654
Stormy        7586
Cloudy        7536
Sandstorms    7495
Windy         7422
Sunny         7284
NaN            616
Name: count, dtype: int64


In [49]:
# Step 1 — Do Mappping to India realistic categories
weather_map = {
    'Sunny'     : 'clear',
    'Cloudy'    : 'clear',
    'Windy'     : 'clear',
    'Fog'       : 'fog',
    'Sandstorms': 'fog',
    'Stormy'    : 'rain'
}

df['weather'] = df['weather'].map(weather_map)

# Step 2 — Fill NaN values with mode 
print("Mode:", df['weather'].mode()[0])
df['weather'].fillna(df['weather'].mode()[0],)

print("\nFinal weather distribution:")
print(df['weather'].value_counts())

Mode: clear

Final weather distribution:
weather
clear    22242
fog      15149
rain      7586
Name: count, dtype: int64


Sunny + Cloudy + Windy → clear
→ Normal delivery conditions

Fog + Sandstorms → fog
→ Visibility kam, time zyada lagta hai

Stormy → rain
→ Surge pricing hoti hai India mein
→ Orders zyada, drivers kam



Excrating festival column

In [50]:

print(df['festival'].value_counts(dropna=False))

# convert festival to binary
df['is_festival'] = df['festival'].apply(
    lambda x: 1 if str(x).strip() == 'Yes' else 0
)

print("Festival distribution:")
print(df['is_festival'].value_counts())

festival
No      44469
Yes       896
NaN       228
Name: count, dtype: int64
Festival distribution:
is_festival
0    44697
1      896
Name: count, dtype: int64


Extracting zone_type

In [51]:

print(df['city'].value_counts(dropna=False))

city
Metropolitian     34093
Urban             10136
NaN                1200
Semi-Urban          164
Name: count, dtype: int64


In [52]:
# Step 1 — Strip and NaN fix
df['city'] = df['city'].str.strip()
df['city'] = df['city'].replace('NaN', np.nan)

# Step 2 — NaN values filled with mode
df['city'] = df['city'].fillna(df['city'].mode()[0])

# Step 3 — Mapping values to busy, normal, low
city_zone_map = {
    'Metropolitian' : 'busy',
    'Urban'         : 'normal',
    'Semi-Urban'    : 'low'
}
df['zone_type'] = df['city'].map(city_zone_map)


print("Zone distribution:")
print(df['zone_type'].value_counts(dropna=False))

Zone distribution:
zone_type
busy      35293
normal    10136
low         164
Name: count, dtype: int64


Normalizing zone_type

In [53]:

def reassign_zone(zone):
    rand = np.random.random()
    if zone == 'busy':
        if rand < 0.40:
            return 'busy'
        elif rand < 0.75:
            return 'normal'
        else:
            return 'low'
    return zone

np.random.seed(42)
df['zone_type'] = df['zone_type'].apply(reassign_zone)

print("New zone distribution:")
print(df['zone_type'].value_counts())
print((df['zone_type'].value_counts(
    normalize=True)*100).round(1))

New zone distribution:
zone_type
normal    22494
busy      14141
low        8958
Name: count, dtype: int64
zone_type
normal    49.3
busy      31.0
low       19.6
Name: proportion, dtype: float64


Why normalizing zone_type?
1. Original data unrealistic tha
2. Model biased ho raha tha

Extracting distance_km column

In [54]:
np.random.seed(42)

def generate_distance(zone):
    if zone == 'busy':
        return round(np.random.uniform(0.5, 4.0), 1)
    elif zone == 'normal':
        return round(np.random.uniform(2.0, 8.0), 1)
    else:
        return round(np.random.uniform(5.0, 15.0), 1)

df['distance_km'] = df['zone_type'].apply(generate_distance)

print("Distance stats:")
print(df['distance_km'].describe())
print("\nSample:")
print(df[['zone_type','distance_km']].head(5))

Distance stats:
count    45593.000000
mean         5.647700
std          4.465554
min          0.500000
25%          1.600000
50%          5.000000
75%          6.500000
max         15.000000
Name: distance_km, dtype: float64

Sample:
  zone_type  distance_km
0    normal          4.2
1       low         14.5
2    normal          6.4
3    normal          5.6
4      busy          1.0


Busy area   → 0.5-4 km
→ Metro cities mein restaurants
  aur customers paas paas hote hain

Normal area → 2-8 km
→ Thodi door delivery hoti hai

Low area    → 5-15 km
→ Semi-urban mein zyada distance
  cover karna padta hai

Making app_name column

In [55]:
np.random.seed(42)

def assign_app(row):
    if row['is_festival'] == 1:
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.45, 0.30, 0.25]
        )
    elif row['weather'] == 'rain':
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.38, 0.37, 0.25]
        )
    elif row['is_lunch_time'] == 1:
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.45, 0.28, 0.27]
        )
    elif row['is_dinner_time'] == 1:
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.28, 0.38, 0.34]
        )
    elif row['is_weekend'] == 1:
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.33, 0.34, 0.33]
        )
    elif row['zone_type'] == 'busy':
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.28, 0.27, 0.45]
        )
    else:
        return np.random.choice(
            ['Zomato', 'Swiggy', 'Blinkit'],
            p=[0.34, 0.33, 0.33]
        )

df['app_name'] = df.apply(assign_app, axis=1)

print("App distribution:")
print(df['app_name'].value_counts())
print("\nPercentages:")
print((df['app_name'].value_counts(
    normalize=True)*100).round(1))

App distribution:
app_name
Swiggy     16833
Zomato     14457
Blinkit    14303
Name: count, dtype: int64

Percentages:
app_name
Swiggy     36.9
Zomato     31.7
Blinkit    31.4
Name: proportion, dtype: float64


Zomato — Festival + Lunch Dominant
→ Zomato har festival pe aggressive
  marketing + driver incentives deta hai
→ Diwali, Holi pe Zomato surge offers
→ Lunch time pe Zomato restaurant
  network zyada strong hai India mein
→ Isliye festival+lunch pe 50% probability
Swiggy — Weekend Dominant
→ Swiggy weekend pe heavy discounts
  aur offers deta hai customers ko
→ Zyada customers = zyada orders
  = zyada driver earnings
→ Isliye weekend pe 45% probability
Blinkit — Busy Zone Dominant
→ Blinkit = grocery + quick commerce
→ 10-minute delivery model
→ Sirf dense urban areas mein
  warehouses hain
→ Busy metro areas mein strongest
→ Isliye busy zone pe 35% probability


Making expected_payout

In [56]:
np.random.seed(42)

def calculate_payout(row):
    base = 40
    distance_pay = row['distance_km'] * 9

    if row['app_name'] == 'Zomato':
        if row['is_lunch_time'] or row['is_dinner_time']:
            app_bonus = 30
        else:
            app_bonus = 12

    elif row['app_name'] == 'Swiggy':
        if row['is_weekend']:
            app_bonus = 25
        else:
            app_bonus = 12

    else:  # Blinkit
        if row['zone_type'] == 'busy':
            app_bonus = 20
        else:
            app_bonus = 10

    weather_surge  = 35 if row['weather'] == 'rain' else 0
    fog_penalty    = -5 if row['weather'] == 'fog'  else 0
    festival_surge = 40 if row['is_festival'] == 1  else 0
    variation      = np.random.uniform(-5, 5)

    total = (base + distance_pay + app_bonus +
             weather_surge + fog_penalty +
             festival_surge + variation)

    return round(max(total, 30), 2)

df['expected_payout'] = df.apply(calculate_payout, axis=1)

print("Payout stats:")
print(df['expected_payout'].describe())
print("\nSample:")
print(df[['app_name','weather',
          'distance_km','expected_payout']].head(5))

Payout stats:
count    45593.000000
mean       111.219757
std         42.659339
min         46.500000
25%         77.730000
50%        102.820000
75%        137.250000
max        264.990000
Name: expected_payout, dtype: float64

Sample:
  app_name weather  distance_km  expected_payout
0   Swiggy   clear          4.2           101.55
1  Blinkit    rain         14.5           220.01
2  Blinkit     fog          6.4           104.92
3   Swiggy   clear          5.6           103.39
4   Zomato   clear          1.0            75.56


Har delivery order ke liye calculate karo:
"Driver ko is order mein kitna paisa milega?"= expected_payout

Base = ₹40
→ Har order pe minimum guaranteed pay
→ Chahe distance kitna bhi ho
→ Zomato/Swiggy India mein
  ₹35-40 base dete hain 

Distance pay = distance × ₹9
→ Har km ke ₹9 milte hain
→ 3 km order → 3 × 9 = ₹27 extra
→ India mein ₹8-10/km realistic hai 

Zomato:
→ Lunch/Dinner pe ₹30 bonus
  Kyun? Peak hours mein Zomato
  drivers ko incentive deta hai
  zyada orders handle karne ke liye

→ Normal time pe ₹12
  Kyun? Off-peak mein bonus kam hota hai

  Swiggy:
→ Weekend pe ₹25 bonus
  Kyun? Swiggy weekend pe
  heavy promotions karta hai
  → Zyada orders → zyada incentive

→ Weekday pe ₹12
  Normal operations

  Blinkit:
→ Busy zone mein ₹20 bonus
  Kyun? Blinkit ka model
  dense areas mein kaam karta hai
  10-min delivery → high incentive

→ Low/Normal zone mein ₹10
  Kyun? Blinkit warehouses
  sirf busy areas mein hain
  → Kam orders = kam incentive

  Rain surge = +₹35
→ Barish mein drivers kam nikalte hain
→ Demand same ya zyada hoti hai
→ Apps surge pricing lagate hain
→ Driver ko zyada milta hai 

Fog penalty = -₹5
→ Fog mein delivery slow hoti hai
→ Time zyada lagta hai
→ Apps thoda kam dete hain
→ Realistic 

Festival surge = +₹40
→ Diwali, Holi pe sabse zyada orders
→ Apps maximum incentive dete hain 

Real life mein exact same nahi hota
Har order thoda alag hota hai

-₹5 to +₹5 random variation
→ Realistic feel aata hai data mein
→ Model better generalize karta hai
→ Bina variation ke data
  too perfect lagta hai 

  Sab add karo:
base + distance + bonus + surges + variation
= total payout

max(total, 30):
→ Minimum ₹30 guarantee
→ Koi bhi order ₹30 se kam nahi
→ India mein minimum order
  payment ₹30+ hoti hai 

  Net profit = Kamaya - Kharch



Making fule_cost,time_cost,net_profit columns

In [57]:
# converting time_taken(min) column to numeric 
df['time_taken(min)'] = pd.to_numeric(
    df['time_taken(min)'].astype(str).str.extract('(\d+)')[0],
    errors='coerce'
)

# Fill NaN values with median
df['time_taken(min)'] = df['time_taken(min)'].fillna(
    df['time_taken(min)'].median()
)

df['fuel_cost'] = (df['distance_km'] * 5).round(2)
df['time_cost'] = (df['time_taken(min)'] * 1.5).round(2)
df['net_profit'] = (
    df['expected_payout'] -
    df['fuel_cost'] -
    df['time_cost']
).round(2)

print("Data type:", df['time_taken(min)'].dtype)
print("\nNet profit stats:")
print(df['net_profit'].describe())
print("\nSample:")
print(df[['expected_payout','fuel_cost',
          'time_cost','net_profit']].head(5))

Data type: int64

Net profit stats:
count    45593.000000
mean        43.539346
std         26.403995
min        -28.900000
25%         24.510000
50%         41.480000
75%         60.990000
max        134.970000
Name: net_profit, dtype: float64

Sample:
   expected_payout  fuel_cost  time_cost  net_profit
0           101.55       21.0       36.0       44.55
1           220.01       72.5       49.5       98.01
2           104.92       32.0       39.0       33.92
3           103.39       28.0       31.5       43.89
4            75.56        5.0       45.0       25.56


Target Variable - best_app

In [58]:
def get_best_app(row):
    profits = {}
    for app in ['Zomato', 'Swiggy', 'Blinkit']:
        base = 40 + row['distance_km'] * 9

        # Base bonuses — equal starting point
        bonuses = {
            'Zomato' : 20,
            'Swiggy' : 20,
            'Blinkit': 20
        }

        # Zomato — lunch time strong
        if app == 'Zomato' and row['is_lunch_time']:
            bonuses['Zomato'] += 15

        # Zomato — festival time strong
        if app == 'Zomato' and row['is_festival']:
            bonuses['Zomato'] += 20

        # Swiggy — weekend  strong
        if app == 'Swiggy' and row['is_weekend']:
            bonuses['Swiggy'] += 12

        # Swiggy — rain  strong
        if app == 'Swiggy' and row['weather'] == 'rain':
            bonuses['Swiggy'] += 10

        # Blinkit — busy zone  strong
        if app == 'Blinkit' and row['zone_type'] == 'busy':
            bonuses['Blinkit'] += 14

        # Blinkit — low zone  okay
        if app == 'Blinkit' and row['zone_type'] == 'low':
            bonuses['Blinkit'] += 8

        # Dinner all three equal 
        if row['is_dinner_time']:
            bonuses['Zomato'] += 8
            bonuses['Swiggy'] += 8
            bonuses['Blinkit'] += 6

        weather_surge  = 35 if row['weather'] == 'rain' else 0
        fog_penalty    = -5 if row['weather'] == 'fog'  else 0
        festival_surge = 40 if row['is_festival'] == 1  else 0

        profit = (base + bonuses[app] +
                  weather_surge + fog_penalty +
                  festival_surge -
                  row['fuel_cost'] - row['time_cost'])

        profits[app] = profit

    return max(profits, key=profits.get)

df['best_app'] = df.apply(get_best_app, axis=1)

print("Best app distribution:")
print(df['best_app'].value_counts())
print("\nPercentages:")
print((df['best_app'].value_counts(
    normalize=True)*100).round(1))

Best app distribution:
best_app
Blinkit    16007
Zomato     15794
Swiggy     13792
Name: count, dtype: int64

Percentages:
best_app
Blinkit    35.1
Zomato     34.6
Swiggy     30.3
Name: proportion, dtype: float64


In [59]:
df = df.rename(
    columns={'time_taken(min)': 'estimated_time_min'}
)

Final Dataset

In [60]:
# Selecting Final columns 
final_columns = [
    'hour', 'day_of_week', 'is_weekend', 'is_festival',
    'weather', 'zone_type', 'is_lunch_time', 'is_dinner_time',
    'distance_km', 'estimated_time_min', 'app_name',
    'expected_payout', 'fuel_cost', 'time_cost',
    'net_profit', 'best_app'
]

# Final dataset 
df_final = df[final_columns].copy()
# Filling Weather NaN with mode 
df_final['weather'] = df_final['weather'].fillna(
    df_final['weather'].mode()[0]
)


print("Shape:", df_final.shape)
print("\nMissing values:")
print(df_final.isnull().sum())
print("\nSample:")
print(df_final.head(3))

df_final.to_csv(
    '../data/processed/gigsmart_dataset.csv',
    index=False
)

print("Dataset saved!")
print(f"Shape: {df_final.shape}")
print(f"Location: data/processed/gigsmart_dataset.csv")

Shape: (45593, 16)

Missing values:
hour                  0
day_of_week           0
is_weekend            0
is_festival           0
weather               0
zone_type             0
is_lunch_time         0
is_dinner_time        0
distance_km           0
estimated_time_min    0
app_name              0
expected_payout       0
fuel_cost             0
time_cost             0
net_profit            0
best_app              0
dtype: int64

Sample:
   hour  day_of_week  is_weekend  is_festival weather zone_type  \
0    11            5           1            0   clear    normal   
1    19            4           0            0    rain       low   
2     8            5           1            0     fog    normal   

   is_lunch_time  is_dinner_time  distance_km  estimated_time_min app_name  \
0              0               0          4.2                  24   Swiggy   
1              0               1         14.5                  33  Blinkit   
2              0               0          6.4          